In [ ]:
import os
import copy
import time
import random
import numpy as np
from pathlib import Path
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.models import efficientnet_v2_s, EfficientNet_V2_S_Weights

# Multi-Class Image Classification


# 이미 Pre-Trained 된 모델을 Transfer Learning 해서 사용

EfficientNetV2-S pretrained on ImageNet 사용

## 불러온 데이터 구조가 이렇다고 치자.

In [ ]:
data/my_dataset/

├── train/
│   ├── class_0/
│   ├── class_1/
│   └── class_2/

├── val/
│   ├── class_0/
│   ├── class_1/
│   └── class_2/

└── test/
    ├── class_0/
    ├── class_1/
    └── class_2/

데이터 로딩 → pretrained model 불러오기 → classifier 교체 → freeze 학습 → fine-tuning → scheduler → label smoothing → early stopping → test 평가 → 모델 저장

## Configuration

In [ ]:
class CFG:
    data_dir = Path("data/my_dataset")  # 가상의 데이터 폴더
    img_size = 224
    batch_size = 32
    num_workers = 4

    # 1단계: backbone freeze 후 classifier만 학습
    freeze_epochs = 5
    freeze_lr = 1e-3

    # 2단계: backbone 일부/전체 unfreeze 후 fine-tuning
    finetune_epochs = 20
    finetune_lr = 1e-4
    weight_decay = 1e-4
    label_smoothing = 0.1
    early_stopping_patience = 7

    model_save_path = "best_efficientnetv2_transfer.pt"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

## Data Transform

In [ ]:
weights = EfficientNet_V2_S_Weights.IMAGENET1K_V1

imagenet_mean = weights.transforms().mean
imagenet_std = weights.transforms().std

train_transform = transforms.Compose
 ([
    transforms.Resize((CFG.img_size, CFG.img_size)),
    # 데이터 증강
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),

    transforms.ColorJitter( brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05 ),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std),

    # 일부 영역을 지워서 overfitting 감소
    transforms.RandomErasing(p=0.25)
])

val_test_transform = transforms.Compose
 ([
    transforms.Resize((CFG.img_size, CFG.img_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std)
])

## Dataset & DataLoader

In [ ]:
train_dir = CFG.data_dir / "train"
val_dir = CFG.data_dir / "val"
test_dir = CFG.data_dir / "test"

train_dataset = datasets.ImageFolder( root=train_dir, transform=train_transform )

val_dataset = datasets.ImageFolder( root=val_dir, transform=val_test_transform )

test_dataset = datasets.ImageFolder( root=test_dir, transform=val_test_transform )

class_names = train_dataset.classes
num_classes = len(class_names)
print("Classes:", class_names)
print("Number of classes:", num_classes)
print("Train size:", len(train_dataset))
print("Val size:", len(val_dataset))
print("Test size:", len(test_dataset))

train_loader = DataLoader(
    train_dataset,
    batch_size=CFG.batch_size,
    shuffle=True,
    num_workers=CFG.num_workers,
    pin_memory=True)

val_loader = DataLoader(
    val_dataset,
    batch_size=CFG.batch_size,
    shuffle=False,
    num_workers=CFG.num_workers,
    pin_memory=True)

test_loader = DataLoader(
    test_dataset,
    batch_size=CFG.batch_size,
    shuffle=False,
    num_workers=CFG.num_workers,
    pin_memory=True)

## Pretrained Model
EfficientNetV2-S pretrained 모델을 불러오고, 마지막 classifier를 내 데이터 클래스 수에 맞게 교체한다.

In [ ]:
def build_model(num_classes):
    model = efficientnet_v2_s(weights=weights)

    # 기존 classifier 구조 확인하려면
      # model.classifier = Sequential( Dropout(...), Linear(in_features, 1000) )

    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential( nn.Dropout(p=0.3), nn.Linear(in_features, num_classes) )
    return model

model = build_model(num_classes)
model = model.to(device)

## Utility Functions

In [ ]:
def freeze_backbone(model): # backbone은 얼리고 classifier만 학습. Transfer Learning의 첫 단계.
    for param in model.features.parameters():
        param.requires_grad = False
    for param in model.classifier.parameters():
        param.requires_grad = True


def unfreeze_backbone(model): # backbone까지 풀어서 전체 fine-tuning. pretrained weight가 망가지지 않게 낮은 learning rate.
    for param in model.parameters():
        param.requires_grad = True


def accuracy_score(logits, labels):
    preds = torch.argmax(logits, dim=1)
    correct = (preds == labels).sum().item()
    total = labels.size(0)
    return correct, total


def train_one_epoch(model, loader, criterion, optimizer, scaler=None):
    model.train()
    running_loss = 0.0
    running_correct = 0
    running_total = 0

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        optimizer.zero_grad()
        # AMP: GPU에서 학습 속도와 메모리 효율 개선
        if scaler is not None:
            with torch.cuda.amp.autocast():
                logits = model(images)
                loss = criterion(logits, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            logits = model(images)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()
        batch_size = images.size(0)
        running_loss += loss.item() * batch_size
        correct, total = accuracy_score(logits, labels)
        running_correct += correct
        running_total += total

    epoch_loss = running_loss / running_total
    epoch_acc = running_correct / running_total
    return epoch_loss, epoch_acc


@torch.no_grad()


def evaluate(model, loader, criterion):
    model.eval()
    running_loss = 0.0
    running_correct = 0
    running_total = 0
    all_preds = []
    all_labels = []

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        logits = model(images)
        loss = criterion(logits, labels)
        batch_size = images.size(0)
        running_loss += loss.item() * batch_size
        preds = torch.argmax(logits, dim=1)
        running_correct += (preds == labels).sum().item()
        running_total += labels.size(0)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    epoch_loss = running_loss / running_total
    epoch_acc = running_correct / running_total
    return epoch_loss, epoch_acc, np.array(all_preds), np.array(all_labels)


def fit( model, train_loader, val_loader, criterion, optimizer, scheduler, epochs, phase_name, patience ):
    best_model_wts = copy.deepcopy(model.state_dict())
    best_val_loss = float("inf")
    best_val_acc = 0.0
    no_improve_count = 0
    use_amp = torch.cuda.is_available()
    scaler = torch.cuda.amp.GradScaler() if use_amp else None
    print(f"\n========== {phase_name} START ==========")

    for epoch in range(1, epochs + 1):
        start_time = time.time()
        train_loss, train_acc = train_one_epoch( model, train_loader, criterion, optimizer, scaler )
        val_loss, val_acc, _, _ = evaluate( model, val_loader, criterion )

        if scheduler is not None:
            scheduler.step(val_loss)

        elapsed = time.time() - start_time
        print(
            f"[{phase_name}] "
            f"Epoch {epoch:02d}/{epochs} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Train Acc: {train_acc:.4f} | "
            f"Val Loss: {val_loss:.4f} | "
            f"Val Acc: {val_acc:.4f} | "
            f"Time: {elapsed:.1f}s"        )

        # Early Stopping 기준: validation loss
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_val_acc = val_acc
            best_model_wts = copy.deepcopy(model.state_dict())
            no_improve_count = 0

            torch.save({
                "model_state_dict": best_model_wts,
                "class_names": class_names,
                "num_classes": num_classes,
                "img_size": CFG.img_size,
                "val_loss": best_val_loss,
                "val_acc": best_val_acc
            }, CFG.model_save_path)
            print(f"  -> Best model saved. Val Loss: {best_val_loss:.4f}")

        else:
            no_improve_count += 1
            print(f"  -> No improvement count: {no_improve_count}/{patience}")

        if no_improve_count >= patience:
            print("Early stopping triggered.")
            break

    model.load_state_dict(best_model_wts)
    print(f"========== {phase_name} END ==========")
    print(f"Best Val Loss: {best_val_loss:.4f}")
    print(f"Best Val Acc : {best_val_acc:.4f}")

    return model

## Loss Function
multi-class classification에서는 CrossEntropyLoss 사용.

model 출력은 softmax 적용 전 logits여야 함.

In [ ]:
criterion = nn.CrossEntropyLoss(
    label_smoothing=CFG.label_smoothing)

# Stage 1 : Freeze Backbone, Train Classifier Only

In [ ]:
freeze_backbone(model)

optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=CFG.freeze_lr,
    weight_decay=CFG.weight_decay )

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=2 )

model = fit(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    epochs=CFG.freeze_epochs,
    phase_name="Freeze Training",
    patience=CFG.early_stopping_patience )

# Stage 2: Unfreeze Backbone, Fine-tuning

In [ ]:
unfreeze_backbone(model)

# fine-tuning에서는 learning rate를 더 작게 잡는다.

optimizer = optim.AdamW(
    model.parameters(),
    lr=CFG.finetune_lr,
    weight_decay=CFG.weight_decay )

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=3  )

model = fit(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    epochs=CFG.finetune_epochs,
    phase_name="Fine-tuning",
    patience=CFG.early_stopping_patience  )

# Test Evaluation

In [ ]:
checkpoint = torch.load(CFG.model_save_path, map_location=device)

model.load_state_dict(checkpoint["model_state_dict"])
model.to(device)

test_loss, test_acc, test_preds, test_labels = evaluate( model, test_loader, criterion )

print("\n========== TEST RESULT ==========")
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Acc : {test_acc:.4f}")

## Accuracy per Class

In [ ]:
class_correct = np.zeros(num_classes)
class_total = np.zeros(num_classes)

for pred, label in zip(test_preds, test_labels):
    class_total[label] += 1
    if pred == label:
        class_correct[label] += 1
print("\n========== PER-CLASS ACCURACY ==========")

for idx, class_name in enumerate(class_names):
    if class_total[idx] == 0:
        acc = 0.0
    else:
        acc = class_correct[idx] / class_total[idx]
    print(f"{class_name}: {acc:.4f} ({int(class_correct[idx])}/{int(class_total[idx])})")

# Inference Function for One Image (추론 함수)

In [ ]:
from PIL import Image

@torch.no_grad()

def predict_one_image(image_path, model, class_names):
    model.eval()
    image = Image.open(image_path).convert("RGB")
    image_tensor = val_test_transform(image).unsqueeze(0).to(device)
    logits = model(image_tensor)
    probs = torch.softmax(logits, dim=1)
    pred_idx = torch.argmax(probs, dim=1).item()
    pred_class = class_names[pred_idx]
    confidence = probs[0, pred_idx].item()

    return pred_class, confidence


'''
사용 예시

pred_class, confidence = predict_one_image( "data/my_dataset/test/class_0/example.jpg", model, class_names  )

print(pred_class, confidence)
'''